# Tutorial 6: State

`State` represents the alignment state of the telescope — the degrees of freedom
(DOFs) such as hexapod positions and bending modes — in one of three **bases**.

`StateFactory` provides a convenient way to create `State` objects with shared
SVD context for basis conversions.

## Three bases

| Basis | Symbol | Description |
|-------|--------|-------------|
| **f** | full | All DOFs, length `n_dof`.  Inactive DOFs are zero. |
| **x** | active | Only the active DOFs, length `n_active`. |
| **v** | SVD | SVD-truncated orthogonal modes, length `n_keep`. Requires `Vh`. |

Conversions between bases are available as properties: `.x`, `.f`, `.v`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

from StarSharp import State, StateFactory

%matplotlib inline

## 1. Construction

`State` is a frozen dataclass.  The minimal construction needs `value` (the
coefficient vector) and `basis`.

In [ ]:
# Full DOF vector (50 DOFs, like Rubin's M2 + camera + bending modes)
n_dof = 50
rng = np.random.default_rng(42)

f_value = np.zeros(n_dof)
f_value[0] = 10.0   # M2 dz
f_value[3] = 0.5    # M2 Rx
f_value[10] = -2.0  # Camera dz

state_f = State(value=f_value, basis="f")
print(f"Basis: {state_f.basis}")
print(f"n_dof: {state_f.n_dof}")
print(f"value[:12]: {state_f.value[:12]}")

In [ ]:
# Active-DOF vector (only selected DOFs)
use_dof = np.array([0, 1, 2, 3, 4, 10, 11, 12, 13, 14])
x_value = rng.normal(0, 1, len(use_dof))

state_x = State(
    value=x_value,
    basis="x",
    use_dof=use_dof,
    n_dof=n_dof,
)
print(f"Basis: {state_x.basis}")
print(f"Active DOFs: {len(use_dof)}")
print(f"x_value: {state_x.value}")

## 2. Basis conversions

### x ↔ f

The `x` (active) basis extracts entries at `use_dof` positions from the `f`
(full) basis.  Converting back zero-fills inactive DOFs.

In [ ]:
# x → f: embed active DOFs into full vector
state_f_from_x = state_x.f
print(f"x → f: basis={state_f_from_x.basis}, length={len(state_f_from_x.value)}")
print(f"Active DOFs filled: {state_f_from_x.value[use_dof]}")
print(f"Inactive DOFs zero: {np.all(np.delete(state_f_from_x.value, use_dof) == 0)}")

In [ ]:
# f → x: extract active DOFs
state_x_back = state_f_from_x.x
print(f"f → x: basis={state_x_back.basis}, length={len(state_x_back.value)}")
print(f"Roundtrip matches: {np.allclose(state_x.value, state_x_back.value)}")

### The v basis: SVD-truncated modes

The `v` basis represents the state in an orthogonal mode basis derived from
SVD of a design matrix.  The transformation is:

$$v = x \cdot V_h^T, \quad x = v \cdot V_h$$

where $V_h$ has shape `(n_keep, n_active)`.

When `n_keep < n_active`, the roundtrip `x → v → x` is **lossy** — only
the component within the kept subspace survives.

In [ ]:
# We'll make a "double zernike" design matrix
# We'll use the full 50 DOFs of Rubin AOS, but say we'll only use the first 5 and
# the 5th is degenerate with the first 4.
kmax = 3
jmax = 11
use_dof = list(range(5))
n_keep = 4

A = np.zeros((kmax+1, jmax+1, n_dof))
for idof, (k, j) in enumerate(
    [(1, 4), (2, 5), (3, 6), (1, 11),]
):
    dz_coef = np.zeros((kmax+1, jmax+1))
    dz_coef[k, j] = 1.0
    A[..., idof] = dz_coef
A[..., 4] = A[..., 0] + A[..., 3]
A = A.reshape(-1, n_dof)

# Create a random design matrix and compute its SVD
U, S, Vh = np.linalg.svd(A[..., use_dof], full_matrices=False)

# Keep only the top 4 modes
Vh_trunc = Vh[:n_keep]
x_value = rng.normal(0, 1, len(use_dof))

# Create a State with Vh for v-basis support
state_with_vh = State(
    value=x_value,
    basis="x",
    use_dof=use_dof,
    n_dof=n_dof,
    Vh=Vh_trunc,
)

print(f"n_dof: {state_with_vh.n_dof}")
print(f"n_active: {state_with_vh.n_active}")
print(f"n_keep: {state_with_vh.n_keep}")
print(f"Vh shape: {Vh_trunc.shape}")

In [ ]:
# x → v
state_v = state_with_vh.v
print(f"v-basis: {state_v.basis}, length={len(state_v.value)}")
print(f"v coefficients: {state_v.value}")

In [ ]:
# v → x → v is lossless
v_rt = state_v.x.v
print(f"v → x → v lossless: {np.allclose(state_v.value, v_rt.value)}")

In [ ]:
# x → v → x is LOSSY when n_keep < len(use_dof)
x_through_v = state_with_vh.v.x
residual = np.linalg.norm(state_with_vh.value - x_through_v.value)
print(f"x → v → x lossy:  {not np.allclose(state_with_vh.value, x_through_v.value)}")
print(f"Residual norm: {residual:.4f}")
print(f"(Only the component in the top-{n_keep} SVD modes survives)")

In [ ]:
# Visualize the lossy roundtrip
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(len(use_dof)) - 0.15, state_with_vh.value, 0.3, label="Original (x)")
ax.bar(np.arange(len(use_dof)) + 0.15, x_through_v.value, 0.3, label=f"After v roundtrip (n_keep={n_keep})")
ax.set_xlabel("Active DOF index")
ax.set_ylabel("Value")
ax.set_title("Lossy x → v → x roundtrip")
ax.legend()

## 3. StateFactory

`StateFactory` computes the SVD of a design matrix and creates `State` objects
with all the metadata needed for seamless basis conversions.

It accepts:
- A numpy array (design matrix)
- An integer (identity matrix)
- A `Sensitivity` object (extracts and flattens the gradient matrix)

In [ ]:
# Create from a design matrix
sf = StateFactory(
    A=A,
    use_dof=use_dof,
    n_keep=n_keep,
)

print(f"n_dof: {sf.n_dof}")
print(f"use_dof: {sf.use_dof}")
print(f"n_keep: {sf.n_keep}")
print(f"Singular values: {sf.S}")
print(f"Vh shape: {sf.Vh.shape}")

In [ ]:
# Factory methods create States with shared SVD context
s_x = sf.from_x(x_value)
print(f"from_x: basis={s_x.basis}, len(s.value)={len(s_x.value)}")

s_f = sf.from_f(np.zeros(n_dof))
print(f"from_f: basis={s_f.basis}, len(s.value)={len(s_f.value)}")

s_v = sf.from_v(np.ones(n_keep))
print(f"from_v: basis={s_v.basis}, len(s.value)={len(s_v.value)}")

In [ ]:
# All three can freely convert between bases
print(f"from_x → v: {sf.from_x(x_value).v.value}")
print(f"from_v → x: {sf.from_v(np.ones(n_keep)).x.value}")
print(f"from_v → f (first 10): {sf.from_v(np.ones(n_keep)).f.value[:10]}")

### DOF selection with string syntax

`StateFactory` accepts a comma-separated string for `use_dof` with range notation.

In [ ]:
# String-based DOF selection
sf_str = StateFactory(
    A=np.eye(50),
    use_dof="0-4,10-14,30-34",
)
print(f"use_dof from string: {sf_str.use_dof}")
print(f"Number of active DOFs: {len(sf_str.use_dof)}")

## 4. State addition

Two `State` objects can be added.  The result is computed in the `f` basis.

In [ ]:
s1 = sf.from_x(rng.normal(0, 1, len(use_dof)))
s2 = sf.from_x(rng.normal(0, 0.5, len(use_dof)))

s_sum = s1 + s2
print(f"Sum basis: {s_sum.basis}")
print(f"Sum matches: {np.allclose(s_sum.value, s1.f.value + s2.f.value)}")

## 5. Visualizing the SVD spectrum

In [ ]:
sf_full = StateFactory(A=A, use_dof=use_dof)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(sf_full.S, 'o-')
ax.axvline(n_keep - 0.5, color='red', ls='--', label=f'n_keep={n_keep}')
ax.set_xlabel("Singular value index")
ax.set_ylabel("Singular value")
ax.set_title("SVD spectrum of the design matrix")
ax.legend()
ax.grid(True, alpha=0.3)

## 6. (Optional) Loading the sensitivity from TS_CONFIG_MTTCS

In [ ]:
# If you have ts_config_mttcs setup, you can create a StateFactory from the sensitivity
# matrix in the config and use the implied SVD states

import os
import yaml
from pathlib import Path


mttcs_dir = Path(os.environ["TS_CONFIG_MTTCS_DIR"])
mtaos_dir = mttcs_dir / "MTAOS/v13/ofc/"

senspath = mtaos_dir / "sensitivity_matrix" / "lsst_sensitivity_dz_31_29_50.yaml"
with open(senspath, "r") as f:
    raw_sens = np.array(yaml.safe_load(f))
normpath = mtaos_dir / "normalization_weights" / "range-fwhm.yaml"
with open(normpath, "r") as f:
    norm = np.array(yaml.safe_load(f))
# Zero out uninteresting terms
raw_sens[0] = 0
raw_sens[:, :4] = 0

sf_full = StateFactory(raw_sens, norm=norm, use_dof="0-9,10-16,30-36", n_keep=12)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sf_full.S)
ax.set_yscale("log")
ax.set_xlabel("Singular value index")
ax.set_ylabel("Singular value")
ax.set_title("SVD spectrum of the design matrix")
ax.legend()
ax.grid(True, alpha=0.3)

## 7. Summary

| Property / Method | Description |
|---|---|
| `State.value` | Coefficient vector |
| `State.basis` | Current basis: `'f'`, `'x'`, or `'v'` |
| `.x`, `.f`, `.v` | Basis conversion properties |
| `.n_keep` | Number of SVD modes (when Vh is set) |
| `use_dof` | Active DOF indices |
| `State + State` | Addition (in f-basis) |
| `StateFactory(A, use_dof, n_keep)` | Factory with SVD context |
| `sf.from_x()`, `.from_f()`, `.from_v()` | Factory state creation |

**Next:** [07_Sensitivity.ipynb](07_Sensitivity.ipynb) — linear sensitivity
of observables to DOF perturbations.